# Question 2B

In [1]:
import subprocess
subprocess.run(["pip", "install", "confluent-kafka"], check=True)

CompletedProcess(args=['pip', 'install', 'confluent-kafka'], returncode=0)

In [2]:
import pandas as pd
import json
import time
from confluent_kafka import Producer, Consumer, KafkaException
from confluent_kafka.admin import AdminClient, NewTopic

In [3]:
BROKER = "localhost:9092"

regions = [
    "Europe",
    "Asia_Pacific",
    "Middle_East",
    "North_America",
    "South_America",
    "Unspecified"
]

admin_client = AdminClient({'bootstrap.servers': BROKER})

existing = admin_client.list_topics(timeout=10).topics
to_delete = [t for t in regions if t in existing]
if to_delete:
    admin_client.delete_topics(to_delete)
    time.sleep(3)
    print(f"Deleted: {to_delete}")

new_topics = [NewTopic(r, num_partitions=1, replication_factor=1) for r in regions]
fs = admin_client.create_topics(new_topics)
for topic, f in fs.items():
    try:
        f.result()
        print(f"Topic created: {topic}")
    except Exception as e:
        print(f"Topic {topic} error: {e}")

Deleted: ['Europe', 'Asia_Pacific', 'Middle_East', 'North_America', 'South_America', 'Unspecified']
Topic created: Europe
Topic created: Asia_Pacific
Topic created: Middle_East
Topic created: North_America
Topic created: South_America
Topic created: Unspecified


In [4]:
def generator(filepaths):
    """Yields one row at a time from each regional CSV file."""
    for filepath in filepaths:
        df = pd.read_csv(filepath)
        for _, row in df.iterrows():
            yield row.to_dict()

In [5]:
def run_producer():
    filepaths = [
        "Europe.csv",
        "Asia_Pacific.csv",
        "Middle_East.csv",
        "North_America.csv",
        "South_America.csv",
        "Unspecified.csv"
    ]

    kafka_producer = Producer({'bootstrap.servers': BROKER})

    count = 0
    for message in generator(filepaths):
        region = str(message.get('Region', 'Unspecified')).replace(' ', '_')
        if region not in regions:
            region = 'Unspecified'

        kafka_producer.produce(
            topic=region,
            value=json.dumps(message, default=str).encode('utf-8')
        )

        count += 1
        if count % 10000 == 0:
            kafka_producer.flush()
            print(f"Sent {count} messages...")

    kafka_producer.flush()
    print(f"Producer done — {count} total messages sent.")

run_producer()

Sent 10000 messages...
Sent 20000 messages...
Sent 30000 messages...
Sent 40000 messages...
Sent 50000 messages...
Sent 60000 messages...
Sent 70000 messages...
Sent 80000 messages...
Sent 90000 messages...
Sent 100000 messages...
Sent 110000 messages...
Sent 120000 messages...
Sent 130000 messages...
Sent 140000 messages...
Sent 150000 messages...
Sent 160000 messages...
Sent 170000 messages...
Sent 180000 messages...
Sent 190000 messages...
Sent 200000 messages...
Sent 210000 messages...
Sent 220000 messages...
Sent 230000 messages...
Sent 240000 messages...
Sent 250000 messages...
Sent 260000 messages...
Sent 270000 messages...
Sent 280000 messages...
Sent 290000 messages...
Sent 300000 messages...
Sent 310000 messages...
Sent 320000 messages...
Sent 330000 messages...
Sent 340000 messages...
Sent 350000 messages...
Sent 360000 messages...
Sent 370000 messages...
Sent 380000 messages...
Sent 390000 messages...
Sent 400000 messages...
Sent 410000 messages...
Sent 420000 messages...
S

In [6]:
def run_consumer():
    kafka_consumer = Consumer({
        'bootstrap.servers': BROKER,
        'group.id':          'retail_group',
        'auto.offset.reset': 'earliest'
    })
    kafka_consumer.subscribe(regions)

    message_count     = {r: 0 for r in regions}
    cumulative_sales  = {r: 0.0 for r in regions}
    unknown_customers = {r: 0 for r in regions}

    empty_polls = 0
    max_empty   = 10  # stop after 10 consecutive empty polls

    while empty_polls < max_empty:
        msg = kafka_consumer.poll(timeout=2.0)

        if msg is None:
            empty_polls += 1
            continue
        if msg.error():
            print(f"Consumer error: {msg.error()}")
            continue

        empty_polls = 0
        topic = msg.topic()
        data  = json.loads(msg.value().decode('utf-8'))

        message_count[topic] += 1

        try:
            sale = float(data.get('Quantity', 0)) * float(data.get('UnitPrice', 0))
        except:
            sale = 0.0
        cumulative_sales[topic] += sale

        customer_id = data.get('CustomerID', None)
        if customer_id in [None, '', 'nan', 'NaN'] or (isinstance(customer_id, float) and pd.isna(customer_id)):
            unknown_customers[topic] += 1

    kafka_consumer.close()

    summary = pd.DataFrame({
        'Topic':             regions,
        'Total_Messages':    [message_count[r] for r in regions],
        'Cumulative_Sales':  [round(cumulative_sales[r], 2) for r in regions],
        'Unknown_Customers': [unknown_customers[r] for r in regions]
    })

    print("\n=== Summary Table ===")
    print(summary.to_string(index=False))
    return summary

summary_df = run_consumer()


=== Summary Table ===
        Topic  Total_Messages  Cumulative_Sales  Unknown_Customers
       Europe          538416        9532618.37             134541
 Asia_Pacific            2431         199563.14                335
  Middle_East             142           4275.73                  2
North_America             442           5397.30                  0
South_America              32           1143.60                  0
  Unspecified             446           4749.79                202


# Question 2C

In [7]:
df = pd.read_csv("Europe.csv")
for region_file in ["Asia_Pacific.csv", "Middle_East.csv", "North_America.csv", "South_America.csv", "Unspecified.csv"]:
    df = pd.concat([df, pd.read_csv(region_file)], ignore_index=True)

df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

print(f"Full dataset: {len(df)} rows")
print(df.head(3))

Full dataset: 541909 rows
  InvoiceNo StockCode                         Description  Quantity  \
0    536365    85123A  WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                 WHITE METAL LANTERN         6   
2    536365    84406B      CREAM CUPID HEARTS COAT HANGER         8   

           InvoiceDate  UnitPrice  CustomerID         Country  Region  \
0  2010-12-01 08:26:00       2.55     17850.0  United Kingdom  Europe   
1  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  Europe   
2  2010-12-01 08:26:00       2.75     17850.0  United Kingdom  Europe   

   TotalPrice  
0       15.30  
1       20.34  
2       22.00  


In [8]:
top3_countries = (
    df.groupby('Country')['InvoiceNo']
    .count()
    .reset_index()
    .rename(columns={'InvoiceNo': 'Number_of_Purchases'})
    .sort_values('Number_of_Purchases', ascending=False)
    .head(3)
)

print("=== Inference and Insights: Top 3 Countries by Number of Purchases ===")
print(top3_countries.to_string(index=False))

=== Inference and Insights: Top 3 Countries by Number of Purchases ===
       Country  Number_of_Purchases
United Kingdom               495478
       Germany                 9495
        France                 8557


In [9]:
top3_names = top3_countries['Country'].tolist()

top3_sales = (
    df[df['Country'].isin(top3_names)]
    .groupby('Country')['TotalPrice']
    .sum()
    .reset_index()
    .rename(columns={'TotalPrice': 'Total_Sales'})
    .sort_values('Total_Sales', ascending=False)
)

top3_sales['Total_Sales'] = top3_sales['Total_Sales'].map('{:,.2f}'.format)

print("=== Inference and Insights: Total Sales for Top 3 Countries ===")
print(top3_sales.to_string(index=False))

=== Inference and Insights: Total Sales for Top 3 Countries ===
       Country  Total_Sales
United Kingdom 8,187,806.36
       Germany   221,698.21
        France   197,403.90


In [10]:
most_sold = (
    df.groupby(['Region', 'Description'])['Quantity']
    .sum()
    .reset_index()
    .sort_values('Quantity', ascending=False)
    .groupby('Region')
    .first()
    .reset_index()
    .rename(columns={'Description': 'Most_Sold_Item', 'Quantity': 'Total_Quantity_Sold'})
)

print("=== Inference and Insights: Most Sold Item per Region ===")
print(most_sold.to_string(index=False))

=== Inference and Insights: Most Sold Item per Region ===
       Region                    Most_Sold_Item  Total_Quantity_Sold
 Asia Pacific                RABBIT NIGHT LIGHT                 5285
       Europe WORLD WAR 2 GLIDERS ASSTD DESIGNS                52742
  Middle East        ICE CREAM SUNDAE LIP GLOSS                   96
North America        RETRO COFFEE MUGS ASSORTED                  504
South America         SMALL HEART FLOWERS HOOK                    24
  Unspecified WORLD WAR 2 GLIDERS ASSTD DESIGNS                   96


# Question 2D

In [19]:
import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.ticker import FuncFormatter
import gc  

gc.collect()

48

In [20]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = ['steelblue', 'darkorange', 'green', 'red', 'purple', 'brown']

# --- Chart 1: Total Messages per Region ---
axes[0].bar(summary_df['Topic'], summary_df['Total_Messages'],
            color=colors, edgecolor='black')
axes[0].set_yscale('log')
axes[0].set_title('Total Messages per Region', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Region', fontsize=10)
axes[0].set_ylabel('Number of Messages', fontsize=10)
axes[0].tick_params(axis='x', rotation=45)

# --- Chart 2: Cumulative Sales per Region ---
axes[1].bar(summary_df['Topic'], summary_df['Cumulative_Sales'],
            color=colors, edgecolor='black')
axes[1].set_yscale('log')
axes[1].set_title('Cumulative Total Sales per Region', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Region', fontsize=10)
axes[1].set_ylabel('Total Sales (£)', fontsize=10)
axes[1].tick_params(axis='x', rotation=45)

# --- Chart 3: Unknown Customers per Region ---
axes[2].bar(summary_df['Topic'], summary_df['Unknown_Customers'],
            color=colors, edgecolor='black')
axes[2].set_yscale('log')
axes[2].set_title('Unknown Customers per Region', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Region', fontsize=10)
axes[2].set_ylabel('Unknown Customers', fontsize=10)
axes[2].tick_params(axis='x', rotation=45)

fig.suptitle('Streaming Summary by Region', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('2B_5_summary_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 2B_5_summary_visualization.png")

Saved: 2B_5_summary_visualization.png


C:\Users\user\AppData\Local\Temp\ipykernel_19988\1295450183.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

colors = ['steelblue', 'darkorange', 'green', 'crimson', 'purple', 'gray']

ax1.bar(summary_df['Topic'], summary_df['Total_Messages'], color=colors, edgecolor='black')
ax1.set_title('Total Messages per Region', fontsize=13, fontweight='bold')
ax1.set_xlabel('Region', fontsize=10)
ax1.set_ylabel('Number of Messages', fontsize=10)
ax1.tick_params(axis='x', rotation=45)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

ax2.bar(summary_df['Topic'], summary_df['Cumulative_Sales'], color=colors, edgecolor='black')
ax2.set_title('Cumulative Sales per Region', fontsize=13, fontweight='bold')
ax2.set_xlabel('Region', fontsize=10)
ax2.set_ylabel('Total Sales (£)', fontsize=10)
ax2.tick_params(axis='x', rotation=45)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))

plt.suptitle('Kafka Consumer Summary', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('2D_1_kafka_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 2D_1_kafka_summary.png")

Saved: 2D_1_kafka_summary.png


C:\Users\user\AppData\Local\Temp\ipykernel_19988\2899076001.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
europe = pd.read_csv("Europe.csv")
others = pd.concat([
    pd.read_csv("Asia_Pacific.csv"),
    pd.read_csv("Middle_East.csv"),
    pd.read_csv("North_America.csv"),
    pd.read_csv("South_America.csv"),
    pd.read_csv("Unspecified.csv")
])
df_small = pd.concat([europe, others])
del europe, others
gc.collect()

df_small['TotalPrice'] = df_small['Quantity'] * df_small['UnitPrice']

top3 = (df_small.groupby('Country')
        .size()
        .sort_values(ascending=False)
        .head(3))
top3_names = top3.index.tolist()
print("Top 3 countries:", top3_names)

top3_sales = (df_small[df_small['Country'].isin(top3_names)]
              .groupby('Country')['TotalPrice']
              .sum()
              .reindex(top3_names))
print("Top 3 sales:\n", top3_sales)

gc.collect()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), dpi=80)

colors_top3 = ['steelblue', 'darkorange', 'green']

ax1.bar(top3.index, top3.values, color=colors_top3, edgecolor='black')
ax1.set_title('Top 3 Countries by Purchases', fontsize=11, fontweight='bold')
ax1.set_xlabel('Country')
ax1.set_ylabel('Number of Purchases')
ax1.tick_params(axis='x', rotation=15)

ax2.bar(top3_sales.index, top3_sales.values, color=colors_top3, edgecolor='black')
ax2.set_title('Total Sales for Top 3 Countries', fontsize=11, fontweight='bold')
ax2.set_xlabel('Country')
ax2.set_ylabel('Total Sales (£)')
ax2.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('2D_2_top3_countries.png', dpi=80, bbox_inches='tight')
plt.close()
gc.collect()
print("Saved: 2D_2_top3_countries.png")

Top 3 countries: ['United Kingdom', 'Germany', 'France']
Top 3 sales:
 Country
United Kingdom    8187806.364
Germany            221698.210
France             197403.900
Name: TotalPrice, dtype: float64
Saved: 2D_2_top3_countries.png


In [16]:
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['steelblue', 'darkorange', 'green', 'crimson', 'purple', 'gray']

bars = ax.barh(most_sold['Region'], most_sold['Total_Quantity_Sold'],
               color=colors, edgecolor='black')

for bar, label in zip(bars, most_sold['Most_Sold_Item']):
    ax.text(bar.get_width() * 0.02, bar.get_y() + bar.get_height() / 2,
            label, va='center', fontsize=9, color='black')

ax.set_title('Most Sold Item per Region', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Quantity Sold', fontsize=11)
ax.set_ylabel('Region', fontsize=11)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig('2D_3_most_sold_per_region.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 2D_3_most_sold_per_region.png")

Saved: 2D_3_most_sold_per_region.png


C:\Users\user\AppData\Local\Temp\ipykernel_19988\1980724870.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
